# Lab 01 - Text Classification

**Name:** _____________________ &nbsp;&nbsp;&nbsp; **Registration No.:** _____________________

In this lab we build a complete text classification pipeline as a refresher of core deep learning concepts before moving on to word embeddings, RNNs/LSTMs, and Transformer-based LLMs later in this course. We will classify short news articles from the **AG News** dataset into one of four topics: *World, Sports, Business,* or *Sci/Tech*. We will compare two classical ML baselines (Naive Bayes, Logistic Regression) against a feedforward neural network trained in PyTorch, all on the same TF-IDF features.

In [ ]:
# If running on a fresh Colab runtime, uncomment the line below
# !pip install -q datasets scikit-learn torch matplotlib pandas numpy

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

from typing import List, Tuple, Dict

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

## Task - 01: Data Loading and Exploration

- Load the AG News dataset (4-class news topic classification).
- Inspect the class distribution and a few sample texts.
- Use a small, fixed-size subset for train and test so the lab runs quickly on a CPU runtime.

Hint: the HuggingFace `datasets` library has this dataset built in: `load_dataset("fancyzhx/ag_news")`. Each example has a `"text"` field and a `"label"` field (0=World, 1=Sports, 2=Business, 3=Sci/Tech).

### Step 1.1: Load the AG News Dataset

In [ ]:
def load_ag_news(train_size: int = 4000, test_size: int = 1000) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Loads the AG News dataset and returns fixed-size, shuffled train and test subsets
    as pandas DataFrames with columns "text" and "label".

    Args:
        train_size (int): number of training examples to sample.
        test_size (int): number of test examples to sample.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: (train_df, test_df)
    """
    dataset = load_dataset("fancyzhx/ag_news")

    train_split = dataset["train"].shuffle(seed=SEED).select(range(train_size))
    test_split = dataset["test"].shuffle(seed=SEED).select(range(test_size))

    train_df = pd.DataFrame({"text": train_split["text"], "label": train_split["label"]})
    test_df = pd.DataFrame({"text": test_split["text"], "label": test_split["label"]})

    return train_df, test_df

### Step 1.2: Explore the Dataset

In [ ]:
def explore_dataset(df: pd.DataFrame, name: str = "train") -> None:
    """
    Prints the shape, class distribution, and a couple of sample rows of the given DataFrame.

    Args:
        df (pd.DataFrame): the dataset split to explore.
        name (str): a label for this split, used in the printed output.

    Returns:
        None
    """
    print(f"--- {name} set ---")
    print(f"Shape: {df.shape}")

    print("\nClass distribution:")
    counts = df["label"].value_counts().sort_index()
    for label, count in counts.items():
        print(f"  {CLASS_NAMES[label]}: {count}")

    print("\nSample rows:")
    print(df.sample(3, random_state=SEED)[["text", "label"]])
    print()

## Task - 02: Preprocessing and Feature Extraction

- Preprocess the raw text (lowercase, strip punctuation/extra whitespace)
- Vectorize the preprocessed text using TF-IDF (`sklearn.feature_extraction.text.TfidfVectorizer`).
- Fit the vectorizer on the training set only, then transform both train and test sets.

### Step 2.1: Text Preprocessing

In [ ]:
def preprocess_text(text: str) -> str:
    """
    Lowercases the text and removes punctuation and extra whitespace.

    Args:
        text (str): raw input text.

    Returns:
        str: cleaned text.
    """
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

### Step 2.2: TF-IDF Vectorization

In [ ]:
def vectorize_text(
    train_texts: List[str],
    test_texts: List[str],
    max_features: int = 5000,
):
    """
    Fits a TfidfVectorizer on the training texts and transforms both train and test texts.

    Args:
        train_texts (List[str]): preprocessed training texts.
        test_texts (List[str]): preprocessed test texts.
        max_features (int): vocabulary size cap for the vectorizer.

    Returns:
        Tuple[scipy.sparse matrix, scipy.sparse matrix, TfidfVectorizer]:
            (X_train_tfidf, X_test_tfidf, fitted_vectorizer)
    """
    vectorizer = TfidfVectorizer(max_features=max_features)
    X_train_tfidf = vectorizer.fit_transform(train_texts)
    X_test_tfidf = vectorizer.transform(test_texts)

    return X_train_tfidf, X_test_tfidf, vectorizer

## Task - 03: Classical Baselines

- Train a **Multinomial Naive Bayes** classifier on the TF-IDF features.
- Train a **Logistic Regression** classifier on the same TF-IDF features.
- These will be your classical ML baselines for comparison against the neural network in Task 4.

### Step 3.1: Train Naive Bayes and Logistic Regression

In [ ]:
def train_naive_bayes(X_train, y_train) -> MultinomialNB:
    """
    Trains a Multinomial Naive Bayes classifier.

    Args:
        X_train: TF-IDF training features.
        y_train: training labels.

    Returns:
        MultinomialNB: the trained classifier.
    """
    model = MultinomialNB()
    model.fit(X_train, y_train)
    return model


def train_logistic_regression(X_train, y_train) -> LogisticRegression:
    """
    Trains a Logistic Regression classifier.

    Args:
        X_train: TF-IDF training features.
        y_train: training labels.

    Returns:
        LogisticRegression: the trained classifier.
    """
    model = LogisticRegression(max_iter=1000, random_state=SEED)
    model.fit(X_train, y_train)
    return model

## Task - 04: Feedforward Neural Network (PyTorch)

Now we refresh the standard deep learning training loop: define a model, a loss function, an optimizer, and iterate over batches doing forward pass → loss → backward pass → optimizer step.

- Build a small `Dataset`/`DataLoader` around the (dense) TF-IDF vectors.
- Define a feedforward network: Linear → ReLU → Dropout → Linear (→ logits over 4 classes).
- Write the training loop using cross-entropy loss and the Adam optimizer.

### Step 4.1: TF-IDF Dataset Wrapper

In [ ]:
class TfidfDataset(Dataset):
    """
    A simple Dataset wrapping dense TF-IDF feature vectors and integer labels.
    """

    def __init__(self, X, y):
        """
        Args:
            X: TF-IDF feature matrix (sparse or dense), shape (n_samples, n_features).
            y: array-like of integer labels, shape (n_samples,).
        """
        X_dense = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
        self.X = torch.tensor(X_dense, dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.long)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        """
        Returns:
            Tuple[torch.FloatTensor, torch.LongTensor]: (feature_vector, label)
        """
        return self.X[idx], self.y[idx]

### Step 4.2: Feedforward Network Architecture

In [ ]:
class TextClassifierFFNN(nn.Module):
    """
    A feedforward neural network for text classification over TF-IDF features.

    Architecture: Linear(input_dim -> hidden_dim) -> ReLU -> Dropout -> Linear(hidden_dim -> num_classes)
    """

    def __init__(self, input_dim: int, hidden_dim: int = 128, num_classes: int = 4, dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): input batch of shape (batch_size, input_dim).

        Returns:
            torch.Tensor: output logits of shape (batch_size, num_classes).
        """
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

### Step 4.3: Training Loop

In [ ]:
def train_ffnn(
    model: nn.Module,
    train_loader: DataLoader,
    num_epochs: int = 10,
    learning_rate: float = 1e-3,
    device: str = "cpu",
) -> nn.Module:
    """
    Trains the given model using cross-entropy loss and the Adam optimizer.

    For each epoch, iterate over train_loader, and for each batch:
        1. Move inputs/labels to `device`.
        2. Zero the optimizer gradients.
        3. Forward pass to get logits.
        4. Compute cross-entropy loss.
        5. Backward pass and optimizer step.
    Print the average loss per epoch.

    Args:
        model (nn.Module): the FFNN to train.
        train_loader (DataLoader): batches of (features, labels).
        num_epochs (int): number of training epochs.
        learning_rate (float): Adam learning rate.
        device (str): "cpu" or "cuda".

    Returns:
        nn.Module: the trained model.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

    return model

## Task - 05: Evaluation and Comparison

- Implement a single evaluation function used consistently across all three models.
- Compute accuracy, macro-averaged precision, recall, and F1-score.
- Plot a confusion matrix for your best-performing model.
- Fill in the Model Comparison Table in the lab document / your report with these numbers.

### Step 5.1: Evaluation Metrics

In [ ]:
def evaluate_predictions(y_true, y_pred) -> Dict[str, float]:
    """
    Computes accuracy and macro-averaged precision, recall, and F1-score.

    Args:
        y_true: ground-truth labels.
        y_pred: predicted labels.

    Returns:
        Dict[str, float]: {"accuracy": ..., "precision": ..., "recall": ..., "f1": ...}
    """
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

### Step 5.2: Feedforward Network Predictions

In [ ]:
def get_ffnn_predictions(model: nn.Module, X_test, device: str = "cpu") -> np.ndarray:
    """
    Runs the trained FFNN over the test features (in eval mode, no gradient tracking)
    and returns the predicted class labels.

    Args:
        model (nn.Module): the trained FFNN.
        X_test: TF-IDF test features (dense or convertible to a dense tensor).
        device (str): "cpu" or "cuda".

    Returns:
        np.ndarray: predicted integer class labels, shape (n_samples,).
    """
    model.eval()
    X_dense = X_test.toarray() if hasattr(X_test, "toarray") else np.asarray(X_test)
    X_tensor = torch.tensor(X_dense, dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(X_tensor)
        preds = torch.argmax(logits, dim=1)

    return preds.cpu().numpy()

### Step 5.3: Confusion Matrix Plot

In [ ]:
def plot_confusion(y_true, y_pred, class_names: List[str], title: str = "Confusion Matrix") -> None:
    """
    Plots a confusion matrix using matplotlib.

    Args:
        y_true: ground-truth labels.
        y_pred: predicted labels.
        class_names (List[str]): display names for each class index.
        title (str): plot title.

    Returns:
        None
    """
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# Task - 06: Putting It All Together

Run the full pipeline end-to-end: load data, preprocess, vectorize, train all three models, evaluate them, and compare. Fill in the Model Comparison Table and Analysis Questions in your report using the numbers printed below.

In [ ]:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Load and explore data
    train_df, test_df = load_ag_news(train_size=4000, test_size=1000)
    explore_dataset(train_df, "train")
    explore_dataset(test_df, "test")

    # 2. Preprocess
    train_texts = [preprocess_text(t) for t in train_df["text"]]
    test_texts = [preprocess_text(t) for t in test_df["text"]]
    y_train = train_df["label"].values
    y_test = test_df["label"].values

    # 3. Vectorize
    X_train_tfidf, X_test_tfidf, vectorizer = vectorize_text(train_texts, test_texts, max_features=5000)
    print(f"TF-IDF feature matrix shape (train): {X_train_tfidf.shape}")

    results = {}

    # 4. Classical baselines
    print("\n--- Training Multinomial Naive Bayes ---")
    nb_model = train_naive_bayes(X_train_tfidf, y_train)
    nb_preds = nb_model.predict(X_test_tfidf)
    results["Naive Bayes"] = evaluate_predictions(y_test, nb_preds)
    print(results["Naive Bayes"])

    print("\n--- Training Logistic Regression ---")
    lr_model = train_logistic_regression(X_train_tfidf, y_train)
    lr_preds = lr_model.predict(X_test_tfidf)
    results["Logistic Regression"] = evaluate_predictions(y_test, lr_preds)
    print(results["Logistic Regression"])

    # 5. Feedforward neural network
    print("\n--- Training Feedforward Neural Network ---")
    train_dataset = TfidfDataset(X_train_tfidf, y_train)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    ffnn = TextClassifierFFNN(input_dim=X_train_tfidf.shape[1], hidden_dim=128, num_classes=4).to(device)
    ffnn = train_ffnn(ffnn, train_loader, num_epochs=10, learning_rate=1e-3, device=device)

    ffnn_preds = get_ffnn_predictions(ffnn, X_test_tfidf, device=device)
    results["Feedforward NN"] = evaluate_predictions(y_test, ffnn_preds)
    print(results["Feedforward NN"])

    # 6. Comparison
    print("\n--- Model Comparison ---")
    comparison_df = pd.DataFrame(results).T
    print(comparison_df)

    # 7. Confusion matrix for the best model (edit which predictions you plot based on your results)
    plot_confusion(y_test, ffnn_preds, CLASS_NAMES, title="Feedforward NN Confusion Matrix")

# Deliverables

Submit your completed `.ipynb` notebook with your name and registration number clearly written at the top, along with a short (1-page) analysis report (`.pdf`) answering the Analysis Questions from the lab document. Please strictly follow the submission guidelines.